# Blackjack simulator

Course project for Python programming. Tier 1, Tier 2 and Tier 3 in one running application.

This notebook is the demo layer. The rules engine, basic strategy chart, Hi-Lo counter, bots and Plotly charts live in the `blackjack` package beside this file, which is also what the unit tests import. Every section below calls the same functions the playable command line table uses, so a number shown here is a number the app can produce.

## How to run

From the project root:

```bash
pip install -r requirements.txt
jupyter notebook notebooks/Prog_Project.ipynb
```

The setup cell adds the current directory to `sys.path`, so opening the notebook from the project root is enough. On Colab, upload the project folder and run the cells in order.

In [1]:
from dataclasses import asdict
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd()
if not (ROOT / "blackjack").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import plotly.io as pio

import blackjack
from blackjack import (
    Analytics,
    BasicStrategyBot,
    BlackjackGame,
    FlatBetBot,
    HiLoCounter,
    HiLoSpreadBot,
    KellyBot,
    MartingaleBot,
    Rules,
    Shoe,
    basic_strategy_action,
    cards_from_text,
    hand_from_cards,
    run_bot,
    run_competition,
    run_simulation,
)
from blackjack import charts
from blackjack.strategy import DEALER_COLUMNS, strategy_rows

try:
    import google.colab  # noqa: F401

    pio.renderers.default = "colab"
except ImportError:
    pio.renderers.default = "notebook"

pd.set_option("display.max_columns", 24)
print("blackjack package", blackjack.__version__)

blackjack package 1.0.0


## Tier 1: core game loop, dealer logic and bankroll

The engine deals from a multi-deck shoe, keeps a bankroll, values aces flexibly, runs the dealer to seventeen, and settles every hand. A natural pays 3:2.

In [2]:
hands = [
    hand_from_cards(["AS", "6H"]),
    hand_from_cards(["AS", "6H", "10D"]),
    hand_from_cards(["AS", "AH", "9D"]),
]
for hand in hands:
    print(f"{hand.display:26} hard={hand.hard_total:2d}  soft={hand.is_soft}")

A♠ 6♥ (17 soft)            hard= 7  soft=True
A♠ 6♥ 10♦ (17)             hard=17  soft=False
A♠ A♥ 9♦ (21 soft)         hard=11  soft=True


In [3]:
shoe = Shoe(6, 0.75, seed=11)
counts = shoe.remaining_counts()
print("cards in shoe:", len(shoe), "| decks:", shoe.num_decks)
print("cut card after", shoe.cut_card_index, "cards")
print("aces:", counts["A"], "| ten-valued:", sum(counts[r] for r in ("10", "J", "Q", "K")))
shoe.draw(10)
print("after ten draws: remaining", shoe.cards_remaining, "| discard", shoe.discard_tray, "| decks left", round(shoe.decks_remaining, 3))

cards in shoe: 312 | decks: 6
cut card after 234 cards
aces: 24 | ten-valued: 96
after ten draws: remaining 302 | discard 10 | decks left 5.808


In [4]:
table = BlackjackGame(Rules(), bankroll=1000, seed=21)
results = table.play_hand(25)
for result in results:
    print(result)
print("bankroll:", table.bankroll, "| round net:", table.round_net)
print("shoe status:", table.shoe_status())

HandResult(hand_index=0, bet=25.0, net=-25.0, outcome=<Outcome.LOSS: 'loss'>, player_total=18, dealer_total=21, is_blackjack=False, is_doubled=False, is_split=False, surrendered=False, start_total=18, dealer_upcard='8', cards=('8♦', 'K♣'), insurance_bet=0.0, insurance_net=0.0)
bankroll: 975.0 | round net: -25.0
shoe status: {'cards_remaining': 307, 'total_cards': 312, 'decks_remaining': 5.9, 'discard_tray': 5, 'penetration': 0.016, 'needs_shuffle': False, 'shuffle_count': 1, 'running_count': -1, 'true_count': -0.17}


In [5]:
natural = BlackjackGame(Rules(), bankroll=1000, seed=1)
natural.shoe.stack(["AS", "9C", "10H", "7D"])
natural.start_hand(25)
result = natural.round_results[0]
print("outcome:", result.outcome.value, "| net:", result.net, "| bankroll:", natural.bankroll)

outcome: blackjack | net: 37.5 | bankroll: 1037.5


In [6]:
def run_pytest(*args):
    command = [sys.executable, "-m", "pytest", "-q", *args]
    completed = subprocess.run(command, capture_output=True, text=True, cwd=ROOT)
    lines = [line for line in completed.stdout.strip().splitlines() if line.strip()]
    print(lines[-1] if lines else completed.stderr[-1500:])
    if completed.returncode != 0:
        print(completed.stdout[-3000:])
    return completed.returncode

run_pytest("tests/test_cards.py", "tests/test_hand.py", "tests/test_engine_basic.py")

80 passed in 0.13s


0

## Tier 2: advanced actions, basic strategy and session analytics

Double, split including re-splitting, surrender and insurance are part of the engine. The strategy module returns the mathematically optimal action for any hand against any dealer upcard. The same live table records decisions, so adherence is measured rather than assumed.

In [7]:
def fresh_table(seed=2):
    return BlackjackGame(Rules(), bankroll=1000, seed=seed)


split = fresh_table(2)
split.shoe.stack(["8S", "9C", "8H", "7D", "3C", "2D", "10C"])
split.start_hand(25)
print("legal actions:", split.available_actions())
split.act("split")
print("after split:", [(hand.display, hand.bet) for hand in split.hands])
split.act("stand")
split.act("stand")
for result in split.round_results:
    print("hand", result.hand_index, result.outcome.value, result.net)
print("bankroll:", split.bankroll)

legal actions: ['hit', 'stand', 'double', 'split', 'surrender']
after split: [('8♠ 3♣ (11)', 25.0), ('8♥ 2♦ (10)', 25.0)]
hand 0 win 25.0
hand 1 win 25.0
bankroll: 1050.0


In [8]:
examples = [
    ("pair of eights", ["8S", "9C", "8H", "7D"]),
    ("soft eighteen", ["AS", "9C", "7H", "7D"]),
    ("hard sixteen", ["10S", "10C", "6H", "7D"]),
]
for label, cards in examples:
    game = fresh_table(3)
    game.shoe.stack(cards)
    game.start_hand(25)
    actions = game.available_actions()
    advice = basic_strategy_action(
        game.player_hand,
        game.dealer.cards[0],
        game.rules,
        can_double="double" in actions,
        can_split="split" in actions,
        can_surrender="surrender" in actions,
    )
    print(f"{label:18} total={game.player_hand.total:2d}  legal={actions}  strategy={advice}")

pair of eights     total=16  legal=['hit', 'stand', 'double', 'split', 'surrender']  strategy=split
soft eighteen      total=18  legal=['hit', 'stand', 'double', 'surrender']  strategy=hit
hard sixteen       total=16  legal=['hit', 'stand', 'double', 'surrender']  strategy=surrender


In [9]:
rows = strategy_rows()
strategy = pd.DataFrame(
    [list(code) for _, code in rows],
    index=[label for label, _ in rows],
    columns=DEALER_COLUMNS,
)
colors = {"H": "#fecaca", "S": "#bbf7d0", "D": "#bfdbfe", "d": "#c7d2fe", "P": "#fde68a"}
print("H hit | S stand | D double else hit | d double else stand | P split")
strategy.style.map(
    lambda value: f"background-color: {colors[value]}; color: #111827; font-weight: 600"
).set_caption("Basic strategy, 6 decks, S17, DAS, late surrender")

H hit | S stand | D double else hit | d double else stand | P split


,2,3,4,5,6,7,8,9,10,A
Hard 5-8,H,H,H,H,H,H,H,H,H,H
Hard 9,H,D,D,D,D,H,H,H,H,H
Hard 10,D,D,D,D,D,D,D,D,H,H
Hard 11,D,D,D,D,D,D,D,D,D,D
Hard 12,H,H,S,S,S,H,H,H,H,H
Hard 13,S,S,S,S,S,H,H,H,H,H
Hard 14,S,S,S,S,S,H,H,H,H,H
Hard 15,S,S,S,S,S,H,H,H,H,H
Hard 16,S,S,S,S,S,H,H,H,H,H
Hard 17,S,S,S,S,S,S,S,S,S,S


In [10]:
session = BlackjackGame(Rules(), bankroll=1000, seed=31)
while session.bankroll >= 10 and session.hands_played < 500:
    session.play_hand(10)
session_stats = Analytics.from_results(
    session.history,
    starting_bankroll=1000,
    decisions=session.decision_history,
)
pd.Series(session_stats.as_dict())

hands                   518.0000
wagered                5800.0000
net                      95.0000
avg_bet                  11.2000
roi_pct                   9.5000
house_edge_pct           -1.6379
wins                    225.0000
losses                  230.0000
pushes                   48.0000
blackjacks               28.0000
surrenders               15.0000
win_rate_pct             43.4400
longest_win_streak        7.0000
longest_loss_streak      10.0000
max_drawdown            265.0000
final_bankroll         1095.0000
strategy_decisions      617.0000
adherence_pct           100.0000
dtype: float64

## Tier 3: shoe penetration, card counting and bots

The shoe uses a cut card and a discard tray. The counter tracks a Hi-Lo running count and converts it to a true count. The bots size their bets from the count, a progression, or a fractional Kelly stake.

In [11]:
penetration_shoe = Shoe(2, 0.2, seed=41)
penetration_game = BlackjackGame(
    Rules(num_decks=2, penetration=0.2),
    bankroll=10000,
    shoe=penetration_shoe,
    log_decisions=False,
)
for _ in range(40):
    if penetration_game.bankroll < 10:
        break
    penetration_game.play_hand(10)
print("shuffles:", penetration_shoe.shuffle_count)
print("cards dealt since last shuffle:", penetration_shoe.cards_dealt)
print("cards remaining:", penetration_shoe.cards_remaining, "| discard tray:", penetration_shoe.discard_tray)
print("penetration reached:", round(penetration_shoe.penetration_actual, 3))

shuffles:

 10
cards dealt since last shuffle: 16
cards remaining: 88 | discard tray: 16
penetration reached: 0.154


In [12]:
counter = HiLoCounter(6)
sequence = cards_from_text("2S 5H 10C AS 7D 9S 3C KH 6D")
rows = []
running = 0
for card in sequence:
    running = counter.observe(card)
    rows.append({"card": str(card), "tag": card.hi_lo_value, "running": running})
print("running count:", counter.running_count, "| true count at five decks:", round(counter.true_count(5), 2))
pd.DataFrame(rows)

running count: 1 | true count at five decks: 0.2


,card,tag,running
0,2♠,1,1
1,5♥,1,2
2,10♣,-1,1
3,A♠,-1,0
4,7♦,0,0
5,9♠,0,0
6,3♣,1,1
7,K♥,-1,0
8,6♦,1,1


### Monte Carlo verification

One hundred thousand rounds each for flat and Hi-Lo make the comparison visible. Both runs use the same seed, so they see the same shoe. The flat line settles near the house edge. The Hi-Lo spread turns that edge negative and ends with a player profit.

In [13]:
flat = run_simulation(
    100000, bet=10, starting_bankroll=1_000_000, seed=7
)
counting = run_bot(
    HiLoSpreadBot(unit=10, spread=8),
    100000,
    starting_bankroll=1_000_000,
    seed=7,
    track_decisions=False,
)
comparison = pd.DataFrame(
    [
        {"strategy": "Flat basic", **flat.analytics().as_dict()},
        {"strategy": "Hi-Lo spread", **counting.analytics().as_dict()},
    ]
).set_index("strategy")
comparison["player_edge_pct"] = -comparison["house_edge_pct"]
comparison[["hands", "avg_bet", "house_edge_pct", "player_edge_pct", "net"]]

,hands,avg_bet,house_edge_pct,player_edge_pct,net
strategy,,,,,
Flat basic,102755,11.03,0.1213,-0.1213,-1375.0
Hi-Lo spread,102755,16.16,-0.5007,0.5007,8315.0


### Interactive dashboard and risk heatmap

In [14]:
charts.dashboard(flat).show()

In [15]:
charts.risk_heatmap(flat).show()

### Card counting against the same shoe

The flat bettor and the Hi-Lo spread see the same cards with the same seed. The only difference is bet sizing and insurance at high counts. A profitable counter should show a lower house edge, and a positive player EV in the high true-count buckets.

In [16]:
comparison = pd.DataFrame(
    [
        {"strategy": "Flat basic", **flat.analytics().as_dict()},
        {"strategy": "Hi-Lo spread", **counting.analytics().as_dict()},
    ]
).set_index("strategy")
comparison["player_edge_pct"] = -comparison["house_edge_pct"]
comparison[["hands", "avg_bet", "house_edge_pct", "player_edge_pct", "net", "max_drawdown"]]

,hands,avg_bet,house_edge_pct,player_edge_pct,net,max_drawdown
strategy,,,,,,
Flat basic,102755,11.03,0.1213,-0.1213,-1375.0,3610.0
Hi-Lo spread,102755,16.16,-0.5007,0.5007,8315.0,6245.0


In [17]:
charts.count_profitability(counting).show()

In [18]:
charts.compare_curves({"Flat basic": flat, "Hi-Lo spread": counting}).show()

### Bot leaderboard

Five autonomous players share one shoe. Each keeps its own bankroll, decisions and drawdown. The leaderboard ranks them by final bankroll.

In [19]:
bots = [
    BasicStrategyBot(name="Basic strategy", unit=10),
    FlatBetBot(name="Flat bettor", unit=10),
    MartingaleBot(name="Martingale", unit=10, max_multiplier=8),
    HiLoSpreadBot(name="Hi-Lo spread", unit=10, spread=8),
    KellyBot(name="Half Kelly", unit=10, fraction=0.5),
]
entries, by_name = run_competition(bots, 1000, starting_bankroll=1000, seed=9)
pd.DataFrame([asdict(entry) for entry in entries])

,rank,name,final_bankroll,net,roi,hands,wins,losses,pushes,max_drawdown,edge,adherence
0,1,Martingale,1675.00,675.00,0.67500,1013,420,455,91,855.00,-0.025196,1.0
1,2,Hi-Lo spread,1075.00,75.00,0.07500,1023,442,452,94,1115.00,-0.004267,1.0
2,3,Flat bettor,810.00,-190.00,-0.19000,1031,437,468,85,485.00,0.016814,1.0
3,4,Half Kelly,382.16,-617.84,-0.61784,1024,414,481,87,662.84,0.054378,1.0
4,5,Basic strategy,370.00,-630.00,-0.63000,1026,412,487,73,820.00,0.055360,1.0


In [20]:
charts.leaderboard_chart(entries, starting_bankroll=1000).show()

## Trainer mode: bet guide and insurance hint

The interactive CLI now gives the trainer advice before the bet prompt. The bet guide uses the visible Hi-Lo count and scales from 1 to 8 units. The count excludes the dealer hole card until it is revealed.

The cell below prints the pre-bet guide, deals a stacked hand with the dealer showing an ace, then prints the insurance hint before the insurance prompt. Below the table it shows the unit ladder.

In [21]:
from io import StringIO
from rich.console import Console
from blackjack import GameState
from blackjack.cli import render_table, _trainer_bet_units, _trainer_true_count

trainer_game = BlackjackGame(Rules(), bankroll=1000, seed=4)

_, pre_tc = _trainer_true_count(trainer_game, True)
pre_units = _trainer_bet_units(pre_tc)
print(f"Pre-bet guide  TC {pre_tc:+.2f} -> {pre_units}u = ${pre_units * 25:,.2f}")

trainer_game.start_hand(25)
_, insurance_tc = _trainer_true_count(trainer_game, False)
if trainer_game.state is GameState.INSURANCE:
    advice = "TAKE" if insurance_tc >= 3 else "DECLINE"
    print(f"Insurance hint  {advice}  (TC {insurance_tc:+.2f} vs +3.00)")

trainer_output = StringIO()
trainer_console = Console(file=trainer_output, width=120, force_terminal=False)
render_table(
    trainer_console,
    trainer_game,
    reveal_dealer=False,
    trainer=True,
    hint=True,
)
print(trainer_output.getvalue())

print("Unit ladder")
for true_count in (-1, 0, 1, 2, 3, 5, 8):
    print(f"  TC {true_count:+d} -> {_trainer_bet_units(true_count)} units")

Pre-bet guide  TC +0.00 -> 1u = $25.00
Insurance hint  DECLINE  (TC +0.17 vs +3.00)


╭────────────┬──────────────────────────────────────────────────────────────────┬────────────┬──────────┬──────────────╮
│ Seat       │ Cards                                                            │        Bet │    Total │ Status       │
├────────────┼──────────────────────────────────────────────────────────────────┼────────────┼──────────┼──────────────┤
│ Dealer     │  A♣ ??                                                           │            │        ? │              │
│   Hand 1   │  5♥ 2♠                                                           │     $25.00 │        7 │ Playing      │
╰────────────┴──────────────────────────────────────────────────────────────────┴────────────┴──────────┴──────────────╯

Bankroll $975.00  |  Shoe 308/312 cards  |  Discard 4  |  Shuffles 1  |  Running +1  |  True +0.17


Unit ladder
  TC -1 -> 1 units
  TC +0 -> 1 units
  TC +1 -> 1 units
  TC +2 -> 2 units
  TC +3 -> 3 units
  TC +5 -> 5 units
  TC +8 -> 8 units


## Validation

The test suite covers chip accounting, payout math, dealer soft-17 behaviour, every cell of the strategy chart, split independence, count accounting, Monte Carlo convergence and the CLI. The slow Monte Carlo tests are excluded here so the notebook stays quick, and run separately with `pytest -m slow`.

In [22]:
run_pytest("-m", "not slow")

1 failed, 509 passed, 3 deselected in 1.26s
33m"blackjack.cli.Confirm.ask", return_value=False):
                code = main(["--seed", "1", "--trainer", "--hint"])
        assert code == 0
        output = capsys.readouterr().out
        assert "Running" in output
        assert "Bet guide" in output
        assert output.index("Bet guide") < output.index("Seat")
        assert "Basic strategy says" in output
    
        answers = iter(["25", "n", "stand", "q"])
        with patch("blackjack.cli.Prompt.ask", side_effect=lambda *a, **k: next(answers)):
            with patch("blackjack.cli.Confirm.ask", return_value=False):
                code = main(["--seed", "4", "--trainer"])
        assert code == 0
        output = capsys.readouterr().out
>       assert "Insurance hint: DECLINE" in output
E       AssertionError: assert 'Insurance hint: DECLINE' in '╔═════════════════════════════════ Table open ═════════════════════════════════╗\n║ \x1bBlackjack\x1b  v1.0.0  ...                 

1

## Requirement map

| Requirement | Where it lives | Demo |
| --- | --- | --- |
| Core loop, aces, hit and stand | `blackjack/engine.py`, `blackjack/hand.py` | Tier 1 cells |
| Dealer S17 and settlement, 3:2 blackjack | `blackjack/engine.py` | Tier 1 cells, `tests/test_engine_basic.py` |
| Bankroll and table UI | `blackjack/cli.py` | `python -m blackjack` |
| Trainer bet guide and insurance hint | `blackjack/cli.py` | Trainer cell |
| Double, split, re-split, surrender, insurance | `blackjack/engine.py` | Tier 2 split cell |
| Basic strategy matrix | `blackjack/strategy.py` | Strategy table cell |
| Session analytics | `blackjack/analytics.py` | Session stats cell |
| Multi-deck shoe, cut card, discard tray | `blackjack/cards.py` | Penetration cell |
| Hi-Lo running and true count | `blackjack/counting.py` | Counting cell |
| Betting bots and leaderboard | `blackjack/bots.py`, `blackjack/simulation.py` | Leaderboard cells |
| Monte Carlo house edge check | `tests/test_simulation.py` | Dashboard, slow test |
| Plotly dashboard, curves, heatmaps | `blackjack/charts.py` | Chart cells |
| PEP8, type hints, docstrings | every module | `black` and `flake8` |

The command line app is the playable surface: `python -m blackjack`. The notebook is the reproducible demonstration and the tests are the proof.